In [0]:
# Objetivo:
# Importar as bibliotecas necessárias
# para leitura, manipulação e
# persistência dos dados.

# Justificativa:
# As bibliotecas importadas serão
# utilizadas ao longo de todo o
# processo de construção da camada
# Silver.

# Ação:
# Importa as bibliotecas utilizadas
# neste notebook.

from pathlib import Path

import pandas as pd

In [0]:
# Objetivo:
# Definir as configurações utilizadas
# durante a execução do notebook.

# Justificativa:
# Centralizar os parâmetros facilita
# a reutilização do pipeline para
# diferentes anos de avaliação e
# aproxima sua implementação de
# ambientes de processamento como
# o Databricks.

# Ação:
# Define o ano de referência e os
# diretórios utilizados pelas camadas
# Bronze e Silver do projeto.

ANO_REFERENCIA = 2023

CAMINHO_BRONZE = (
    Path("dados")
    / "bronze"
    / "metas_municipios"
)

CAMINHO_SILVER = (
    Path("dados")
    / "silver"
    / "metas_municipios"
)

# 1. Auditoria da Fonte de Dados - Base Metas por Município

> **Nota**
>
> Durante o desenvolvimento deste projeto foi utilizado o dicionário oficial
> dos Microdados da Avaliação da Alfabetização disponibilizado pelo INEP como
> referência para interpretação das variáveis, domínios e regras de negócio
> presentes nas bases de dados.

## 1.1 Leitura das Bases

**Contexto**

Os dados da entidade **Metas por Município** foram disponibilizados pelo
INEP em arquivos no formato Excel, separados por ano e organizados em
múltiplas abas.

Nesta etapa, os arquivos são carregados para o ambiente de análise,
preservando sua estrutura original para que seja possível realizar a
auditoria da qualidade dos dados antes da aplicação das transformações da
camada Silver.

**Objetivo**

Realizar a leitura das bases da entidade Metas por Município referentes
aos anos de 2023, 2024 e 2025.

**Resultado esperado**

Disponibilizar os dados dos três anos em DataFrames independentes,
mantendo a estrutura original dos arquivos da camada Bronze para as etapas
de auditoria e transformação da camada Silver.

In [0]:
# Objetivo:
# Realizar a leitura das bases da
# entidade Metas por Município
# armazenadas na camada Bronze.

# Justificativa:
# A leitura das bases preserva os
# dados originais disponibilizados
# pelo INEP, permitindo que todas as
# etapas de auditoria sejam realizadas
# antes das transformações da camada
# Silver.

# Ação:
# Carrega a aba "Divulgação Alfabet
# Municipio" das bases referentes aos
# anos de 2023, 2024 e 2025.

df_metas_municipios_2023 = pd.read_excel(
    CAMINHO_BRONZE
    / "ano=2023"
    / "metas_municipios_2023.xlsx",
    sheet_name="Divulgação Alfabet Municipio",
    header=1
)

df_metas_municipios_2024 = pd.read_excel(
    CAMINHO_BRONZE
    / "ano=2024"
    / "metas_municipios_2024.xlsx",
    sheet_name="Divulgação Alfabet Municipio",
    header=1
)

df_metas_municipios_2025 = pd.read_excel(
    CAMINHO_BRONZE
    / "ano=2025"
    / "metas_municipios_2025.xlsx",
    sheet_name="Divulgação Alfabet Municipio",
    header=1
)

In [0]:
# Objetivo:
# Remover registros que não fazem
# parte da base de dados.

# Justificativa:
# Os arquivos do INEP possuem, ao
# final da planilha, linhas contendo
# observações textuais que não
# representam registros da tabela.
# Sua remoção evita interferências na
# inferência dos tipos de dados e nas
# etapas de auditoria da camada
# Silver.

# Ação:
# Remove as linhas cujo campo ANO não
# representa um valor numérico.

def remover_observacoes(df):

    return (
        df[
            pd.to_numeric(
                df["ANO"],
                errors="coerce"
            ).notna()
        ]
        .copy()
    )


df_metas_municipios_2023 = remover_observacoes(
    df_metas_municipios_2023
)

df_metas_municipios_2024 = remover_observacoes(
    df_metas_municipios_2024
)

df_metas_municipios_2025 = remover_observacoes(
    df_metas_municipios_2025
)

## 1.2 Inspeção Inicial da Estrutura

**Contexto**

Após o carregamento das bases da camada Bronze, realiza-se uma inspeção
inicial para compreender a estrutura dos dados disponibilizados pelo INEP
em cada ano da avaliação.

Essa verificação permite identificar a quantidade de colunas e possíveis
alterações estruturais entre as bases antes do processo de padronização da
camada Silver.

**Objetivo**

Inspecionar a estrutura das bases da entidade Metas por Município
referentes aos anos de 2023, 2024 e 2025, identificando eventuais
diferenças entre os esquemas disponibilizados pelo INEP.

**Resultado esperado**

Obter uma visão inicial da estrutura das bases, permitindo identificar
alterações entre os anos e subsidiar as etapas de auditoria e
padronização da camada Silver.

In [0]:
# Objetivo:
# Realizar uma inspeção inicial da
# estrutura das bases de Metas por
# Município.

# Justificativa:
# A inspeção inicial permite verificar
# o esquema das bases e identificar
# possíveis alterações nas colunas
# disponibilizadas pelo INEP ao longo
# dos anos da avaliação.

# Ação:
# Exibe a estrutura das bases de
# Metas por Município para comparação
# entre os anos de 2023, 2024 e 2025.

print(df_metas_municipios_2023.columns.tolist())

print(df_metas_municipios_2024.columns.tolist())

print(df_metas_municipios_2025.columns.tolist())

## 1.3 Comparação dos Tipos de Dados

**Contexto**

Além da estrutura das bases, é importante verificar se os tipos de dados
das colunas permaneceram consistentes entre os anos da avaliação.

Alterações nos tipos podem impactar as etapas de transformação,
padronização e integração dos dados na camada Silver.

**Objetivo**

Comparar os tipos de dados das colunas presentes nas bases da entidade
Metas por Município referentes aos anos de 2023, 2024 e 2025.

**Resultado esperado**

Identificar possíveis divergências entre os tipos de dados das colunas,
subsidiando a definição da estrutura padronizada da camada Silver.

In [0]:
# Objetivo:
# Comparar os tipos de dados das
# bases de Metas por Município dos
# anos de 2023, 2024 e 2025.

# Justificativa:
# Colunas com o mesmo nome podem
# apresentar tipos diferentes entre
# os anos ou novas variáveis podem
# ter sido incorporadas pelo INEP.

# Ação:
# Consolida os tipos de dados das
# três bases em um único relatório
# e classifica a compatibilidade
# entre os esquemas.

relatorio_dtypes = pd.DataFrame({
    "2023": df_metas_municipios_2023.dtypes.astype(str),
    "2024": df_metas_municipios_2024.dtypes.astype(str),
    "2025": df_metas_municipios_2025.dtypes.astype(str)
})

relatorio_dtypes = relatorio_dtypes.replace("nan", pd.NA)


def classificar_status(linha):

    if (
        pd.notna(linha["2023"])
        and pd.isna(linha["2024"])
        and pd.isna(linha["2025"])
    ):
        return "Exclusiva de 2023"

    if (
        pd.isna(linha["2023"])
        and pd.notna(linha["2024"])
        and pd.notna(linha["2025"])
    ):
        return "Nova a partir de 2024"

    if (
        pd.isna(linha["2023"])
        and pd.isna(linha["2024"])
        and pd.notna(linha["2025"])
    ):
        return "Nova em 2025"

    tipos = linha.dropna()

    if len(tipos.unique()) == 1:
        return "Compatível"

    return "Divergente"


relatorio_dtypes["status"] = relatorio_dtypes.apply(
    classificar_status,
    axis=1
)

display(relatorio_dtypes)

In [0]:
# Objetivo:
# Identificar valores que impedem
# o reconhecimento das colunas como
# numéricas.

# Justificativa:
# Embora a amostra apresente apenas
# valores numéricos, algumas colunas
# foram classificadas como object.
# É necessário verificar se existem
# registros incompatíveis com o tipo
# numérico.

# Ação:
# Exibe os valores que não podem ser
# convertidos para número.

colunas = [
    "ANO",
    "CO_UF",
    "CO_MUNICIPIO",
    "META_FINAL_2024",
    "META_FINAL_2025",
    "META_FINAL_2026",
    "META_FINAL_2027",
    "META_FINAL_2028",
    "META_FINAL_2029",
    "META_FINAL_2030",
    "PC_AVALIADOS_LP"
]

for coluna in colunas:

    print(f"\n{coluna}")

    for ano, df in [
        (2023, df_metas_municipios_2023),
        (2024, df_metas_municipios_2024),
        (2025, df_metas_municipios_2025)
    ]:

        invalidos = df.loc[
            pd.to_numeric(
                df[coluna],
                errors="coerce"
            ).isna(),
            coluna
        ].unique()

        print(f"{ano}: {invalidos}")

## 1.4 Análise dos Valores Ausentes

**Contexto**

Após a verificação da estrutura e dos tipos de dados, torna-se necessário
avaliar a presença de valores ausentes nas bases dos diferentes anos.

Essa análise permite identificar possíveis impactos sobre a qualidade dos
dados e definir se será necessário realizar algum tratamento antes da
persistência da camada Silver.

**Objetivo**

Identificar e quantificar a ocorrência de valores ausentes nas bases da
entidade Metas por Município, avaliando a necessidade de tratamento
durante o processo de transformação.

**Resultado esperado**

Obter um diagnóstico da completude dos dados, permitindo justificar as
decisões adotadas em relação ao tratamento de valores ausentes na camada
Silver.

In [0]:
# Objetivo:
# Analisar a ocorrência de valores
# ausentes nas bases da entidade
# Metas por Município.

# Justificativa:
# As bases utilizam o caractere "-"
# para representar ausência de
# informação em algumas colunas.
# Antes da análise, esse marcador é
# convertido para valores ausentes
# reconhecidos pelo Pandas.

# Ação:
# Substitui o marcador "-" por valores
# ausentes e calcula a quantidade de
# valores ausentes por coluna em cada
# base.

for ano, df in [
    (2023, df_metas_municipios_2023),
    (2024, df_metas_municipios_2024),
    (2025, df_metas_municipios_2025)
]:

    print(f"\nValores ausentes - {ano}")

    df_analise = df.replace("-", pd.NA)

    valores_ausentes = (
        df_analise
        .isna()
        .sum()
        .loc[lambda s: s > 0]
        .sort_values(ascending=False)
        .to_frame("Valores Ausentes")
    )

    if valores_ausentes.empty:
        print("Nenhum valor ausente encontrado.")
    else:
        display(valores_ausentes)

In [0]:
# Objetivo:
# Verificar se os valores ausentes
# identificados pertencem aos mesmos
# municípios.

# Justificativa:
# A análise permite verificar se os
# valores ausentes representam uma
# ausência de resultados para um
# mesmo conjunto de municípios ou se
# ocorrem de forma independente entre
# as colunas.

# Ação:
# Exibe os municípios que apresentam
# valores ausentes nas colunas de
# percentual de alfabetização e metas.

colunas_2023 = [
    "CO_MUNICIPIO",
    "NO_MUNICIPIO",
    "PC_ALUNO_ALFABETIZADO",
    "META_FINAL_2024",
    "META_FINAL_2025",
    "META_FINAL_2026",
    "META_FINAL_2027",
    "META_FINAL_2028",
    "META_FINAL_2029",
    "META_FINAL_2030",
    "NIVEIS_ALFABETIZACAO_2023",
    "PC_AVALIADOS_LP"
]

display(
    df_metas_municipios_2023
        .replace("-", pd.NA)
        .loc[
            lambda df:
                df["PC_ALUNO_ALFABETIZADO"].isna(),
            colunas_2023
        ]
)

In [0]:
# Objetivo:
# Verificar se os valores ausentes
# identificados nas bases de 2024 e
# 2025 pertencem ao mesmo conjunto
# de municípios.

# Justificativa:
# Caso os valores ausentes ocorram
# simultaneamente para um mesmo
# conjunto de municípios, isso indica
# uma característica da divulgação
# oficial do INEP e não uma
# inconsistência dos dados.

# Ação:
# Exibe os municípios que apresentam
# ausência de informações nas bases
# de 2024 e 2025.

colunas_2024 = [
    "CO_MUNICIPIO",
    "NO_MUNICIPIO",
    "PC_ALUNO_ALFABETIZADO_2023",
    "PC_ALUNO_ALFABETIZADO_2024",
    "META_FINAL_2024"
]

display(
    df_metas_municipios_2024
        .replace("-", pd.NA)
        .loc[
            lambda df:
                df["PC_ALUNO_ALFABETIZADO_2023"].isna(),
            colunas_2024
        ]
)

colunas_2025 = [
    "CO_MUNICIPIO",
    "NO_MUNICIPIO",
    "PC_ALUNO_ALFABETIZADO_2023",
    "PC_ALUNO_ALFABETIZADO_2024",
    "PC_ALUNO_ALFABETIZADO_2025",
    "META_FINAL_2024",
    "META_FINAL_2025"
]

display(
    df_metas_municipios_2025
        .replace("-", pd.NA)
        .loc[
            lambda df:
                (
                    df["PC_ALUNO_ALFABETIZADO_2023"].isna()
                    | df["PC_ALUNO_ALFABETIZADO_2024"].isna()
                    | df["PC_ALUNO_ALFABETIZADO_2025"].isna()
                ),
            colunas_2025
        ]
)

### 1.4.1 Análise dos Resultados

A análise identificou a ocorrência de valores ausentes nas bases dos anos
de 2023, 2024 e 2025.

A investigação realizada mostrou que essas ausências não representam
inconsistências na qualidade dos dados, mas refletem as regras de divulgação
estabelecidas pelo INEP para os resultados da Avaliação da Alfabetização.

Foi observado que os municípios podem apresentar resultados em alguns anos
e ausência de informações em outros, evidenciando que cada coluna de
percentual de alunos alfabetizados representa exclusivamente o resultado do
respectivo ano de avaliação.

Esse comportamento está alinhado às observações disponibilizadas pelo
próprio INEP, segundo as quais municípios sem participação na avaliação ou
com percentual de participação inferior ao mínimo estabelecido não possuem
resultados divulgados.

Dessa forma, os valores ausentes serão preservados durante a construção da
camada Silver, não sendo aplicadas técnicas de imputação ou exclusão de
registros, por representarem uma característica da fonte oficial de dados.

## 1.5 Seleção das Colunas da Camada Silver

**Contexto**

A auditoria das bases evidenciou que o INEP evoluiu a estrutura dos dados
da entidade Metas por Município ao longo dos anos, incorporando novas
colunas para representar o histórico dos percentuais de alunos
alfabetizados por ano de avaliação.

Além disso, foram identificadas alterações de nomenclatura em algumas
variáveis, mantendo, entretanto, o mesmo significado de negócio.

Nesta etapa é realizada a padronização da estrutura das bases, preservando
a evolução do esquema e garantindo compatibilidade entre as partições da
camada Silver.

**Objetivo**

Definir o conjunto de atributos que comporá a Base Metas por Município da
camada Silver, padronizando a estrutura das bases de 2023, 2024 e 2025.

**Resultado esperado**

Obter três bases com a mesma estrutura de colunas, preservando a evolução
histórica dos indicadores e garantindo compatibilidade estrutural para as
etapas posteriores do pipeline analítico.

In [0]:
# Objetivo:
# Selecionar as colunas que farão
# parte da Base Metas por Município
# da camada Silver.

# Justificativa:
# As bases sofreram evolução de
# estrutura entre 2023 e 2025,
# incorporando novos indicadores e
# alterando a nomenclatura de algumas
# colunas. A padronização preserva a
# evolução histórica e garante um
# esquema único para todas as
# partições.

# Ação:
# Padroniza os nomes das colunas,
# cria os atributos ausentes nas
# bases anteriores, converte o
# marcador "-" para valores ausentes
# e seleciona o conjunto final de
# atributos da camada Silver.

# Padronização dos nomes das colunas
df_metas_municipios_2023 = (
    df_metas_municipios_2023
    .rename(columns={
        "PC_ALUNO_ALFABETIZADO": "PC_ALUNO_ALFABETIZADO_2023",
        "NIVEIS_ALFABETIZACAO_2023": "CO_NIVEL_ALFABETIZACAO"
    })
)

# Estrutura final da camada Silver
colunas_silver = [
    "ANO",
    "CO_UF",
    "SG_UF",
    "CO_MUNICIPIO",
    "NO_MUNICIPIO",
    "NO_TP_REDE",
    "PC_ALUNO_ALFABETIZADO_2023",
    "PC_ALUNO_ALFABETIZADO_2024",
    "PC_ALUNO_ALFABETIZADO_2025",
    "META_FINAL_2024",
    "META_FINAL_2025",
    "META_FINAL_2026",
    "META_FINAL_2027",
    "META_FINAL_2028",
    "META_FINAL_2029",
    "META_FINAL_2030",
    "CO_NIVEL_ALFABETIZACAO",
    "PC_AVALIADOS_LP"
]

# Padroniza as três bases
for df in [
    df_metas_municipios_2023,
    df_metas_municipios_2024,
    df_metas_municipios_2025
]:

    df.replace("-", pd.NA, inplace=True)

    for coluna in colunas_silver:

        if coluna not in df.columns:
            df[coluna] = pd.NA

# Seleciona apenas as colunas da Silver
df_metas_municipios_2023 = (
    df_metas_municipios_2023[colunas_silver]
    .copy()
)

df_metas_municipios_2024 = (
    df_metas_municipios_2024[colunas_silver]
    .copy()
)

df_metas_municipios_2025 = (
    df_metas_municipios_2025[colunas_silver]
    .copy()
)

## 1.6 Validação da Estrutura da Camada Silver

**Contexto**

Após a padronização da estrutura das bases, torna-se necessário verificar
se as três partições anuais compartilham exatamente o mesmo esquema.

Essa validação garante que todas as bases estejam preparadas para a
persistência na camada Silver e para sua utilização nas etapas posteriores
do pipeline analítico.

**Objetivo**

Validar a estrutura das bases da entidade Metas por Município após o
processo de padronização.

**Resultado esperado**

Confirmar que as bases de 2023, 2024 e 2025 possuem exatamente a mesma
quantidade de colunas, assegurando a compatibilidade estrutural entre as
partições da camada Silver.

In [0]:
# Objetivo:
# Validar a estrutura das bases da
# entidade Metas por Município após
# o processo de padronização.

# Justificativa:
# A validação confirma que todas as
# partições da camada Silver possuem
# exatamente o mesmo esquema antes
# da persistência dos dados.

# Ação:
# Compara a quantidade de colunas das
# três bases e classifica o resultado
# da validação.

validacao = pd.DataFrame({
    "Base": ["2023", "2024", "2025"],
    "Quantidade de Colunas": [
        len(df_metas_municipios_2023.columns),
        len(df_metas_municipios_2024.columns),
        len(df_metas_municipios_2025.columns)
    ]
})

quantidade_esperada = len(colunas_silver)

validacao["Status"] = validacao[
    "Quantidade de Colunas"
].apply(
    lambda x: (
        "Compatível"
        if x == quantidade_esperada
        else "Incompatível"
    )
)

display(validacao)

## 1.7 Persistência da Base Metas por Município

**Contexto**

Após a validação da estrutura das bases, os dados encontram-se prontos para
serem persistidos na camada Silver.

Nesta etapa, cada partição anual é armazenada em seu respectivo diretório,
preservando a organização por entidade e por ano definida para a
arquitetura do projeto.

**Objetivo**

Persistir as bases da entidade Metas por Município na camada Silver,
mantendo o particionamento anual.

**Resultado esperado**

Armazenar as bases tratadas da entidade Metas por Município na camada
Silver, preservando a estrutura padronizada e a organização do Data Lake
para as etapas posteriores do pipeline analítico.

In [0]:
# Objetivo:
# Persistir as bases da entidade
# Metas por Município na camada
# Silver.

# Justificativa:
# A persistência organiza os dados
# tratados em partições anuais,
# facilitando sua reutilização nas
# etapas posteriores do pipeline.

# Ação:
# Cria os diretórios da camada Silver,
# quando necessário, e grava as bases
# tratadas em arquivos CSV.

for ano, df in [
    (2023, df_metas_municipios_2023),
    (2024, df_metas_municipios_2024),
    (2025, df_metas_municipios_2025)
]:

    destino = (
        CAMINHO_SILVER
        / f"ano={ano}"
    )

    destino.mkdir(
        parents=True,
        exist_ok=True
    )

    df.to_csv(
        destino
        / f"TS_METAS_MUNICIPIOS_{ano}_SILVER.csv",
        sep=";",
        decimal=",",
        encoding="utf-8",
        index=False
    )

# Conclusão

Ao longo deste notebook foi realizada a auditoria estrutural das bases da
entidade **Metas por Município** referentes aos anos de **2023**, **2024**
e **2025**, identificando tanto a evolução do esquema de dados quanto as
regras de divulgação adotadas pelo INEP.

A auditoria contemplou a verificação da estrutura das bases, a comparação
dos tipos de dados e a análise dos valores ausentes. Durante esse processo,
foi identificado que parte das ausências era representada pelo caractere
`"-"` nos arquivos originais. Após investigação, verificou-se que esse
marcador corresponde à ausência de resultados divulgados pelo INEP,
decorrente de municípios que não participaram da avaliação ou que não
atingiram o percentual mínimo de participação estabelecido.

Também foi identificada a evolução do modelo de dados ao longo dos anos,
com a substituição do indicador único de percentual de alunos
alfabetizados por uma estrutura que preserva os resultados históricos de
cada edição da avaliação. Para garantir a compatibilidade entre as
partições anuais, foi realizada a padronização das nomenclaturas e da
estrutura das colunas, preservando integralmente o significado dos dados
originais.

Além da padronização estrutural, o marcador textual de ausência de dados
foi convertido para valores ausentes reconhecidos pelo Pandas (`pd.NA`),
mantendo a semântica da informação e permitindo um tratamento consistente
nas etapas analíticas posteriores.

Após a validação da estrutura, as bases foram persistidas na camada
**Silver**, mantendo a organização por entidade e o particionamento por
ano, conforme a arquitetura definida para o projeto.

Dessa forma, a Base Metas por Município da camada Silver passa a
representar uma versão padronizada, auditada e consistente dos dados
oficiais disponibilizados pelo INEP, preservando sua evolução histórica e
fornecendo uma base confiável para a construção dos indicadores e análises
desenvolvidos na camada Gold.